# 第 24 章：异常检测与新奇发现

这个 notebook 用一个教学版光谱特征数据集演示：

- 最近邻距离分数如何做最朴素的异常排序
- LOF-like 分数如何强调局部密度差异
- 简化版 isolation score 如何把容易被隔离的点顶出来
- 为什么高分样本还需要人工复核


In [ ]:
from __future__ import annotations

import csv
import math
import random
from pathlib import Path

DATA_PATH = Path("../../data/small/spectral_anomaly_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            "spectrum_id": raw["spectrum_id"],
            "label": raw["label"],
            "halpha_ew": float(raw["halpha_ew"]),
            "caii_k_ew": float(raw["caii_k_ew"]),
            "blue_red_ratio": float(raw["blue_red_ratio"]),
            "snr": float(raw["snr"]),
            "reference_flag": raw["reference_flag"],
        })

print(f"Loaded {len(rows)} spectral samples from {DATA_PATH.name}")
flag_counts = {}
for row in rows:
    flag_counts[row["reference_flag"]] = flag_counts.get(row["reference_flag"], 0) + 1
print("reference flags:", flag_counts)
print("first row:", rows[0])


In [ ]:
feature_names = ["halpha_ew", "caii_k_ew", "blue_red_ratio", "snr"]
means = {}
stds = {}
for name in feature_names:
    values = [row[name] for row in rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    means[name] = mean_value
    stds[name] = math.sqrt(variance) or 1.0

standardized_rows = []
for row in rows:
    scaled = {name: (row[name] - means[name]) / stds[name] for name in feature_names}
    standardized_rows.append({**row, **scaled})

print("means:", {name: round(value, 3) for name, value in means.items()})
print("stds:", {name: round(value, 3) for name, value in stds.items()})
print("known unusual ids:", [row["spectrum_id"] for row in standardized_rows if row["reference_flag"] != "normal"])


In [ ]:
def feature_vector(row):
    return [row[name] for name in feature_names]


def euclidean_distance(left_vector, right_vector):
    return math.sqrt(sum((left - right) ** 2 for left, right in zip(left_vector, right_vector)))


vectors = {row["spectrum_id"]: feature_vector(row) for row in standardized_rows}
k_neighbors = 3

mean_neighbor_distance = {}
nearest_neighbors = {}
for row in standardized_rows:
    source_id = row["spectrum_id"]
    distances = []
    for other in standardized_rows:
        other_id = other["spectrum_id"]
        if other_id == source_id:
            continue
        distances.append((euclidean_distance(vectors[source_id], vectors[other_id]), other_id))
    distances.sort(key=lambda item: item[0])
    nearest_neighbors[source_id] = [other_id for _, other_id in distances[:k_neighbors]]
    mean_neighbor_distance[source_id] = sum(distance for distance, _ in distances[:k_neighbors]) / k_neighbors

distance_ranking = sorted(mean_neighbor_distance.items(), key=lambda item: item[1], reverse=True)
print("Top candidates by mean neighbor distance:")
for source_id, score in distance_ranking[:5]:
    row = next(item for item in standardized_rows if item["spectrum_id"] == source_id)
    print(source_id, round(score, 3), row["reference_flag"], nearest_neighbors[source_id])


In [ ]:
k_distance = {}
for source_id in vectors:
    ordered = sorted(
        euclidean_distance(vectors[source_id], vectors[other_id])
        for other_id in vectors
        if other_id != source_id
    )
    k_distance[source_id] = ordered[k_neighbors - 1]


def reachability_distance(left_id, right_id):
    raw_distance = euclidean_distance(vectors[left_id], vectors[right_id])
    return max(k_distance[right_id], raw_distance)


local_reachability_density = {}
for source_id in vectors:
    average_reachability = sum(
        reachability_distance(source_id, neighbor_id)
        for neighbor_id in nearest_neighbors[source_id]
    ) / k_neighbors
    local_reachability_density[source_id] = 1.0 / average_reachability

lof_like_score = {}
for source_id in vectors:
    lof_like_score[source_id] = sum(
        local_reachability_density[neighbor_id] / local_reachability_density[source_id]
        for neighbor_id in nearest_neighbors[source_id]
    ) / k_neighbors

lof_ranking = sorted(lof_like_score.items(), key=lambda item: item[1], reverse=True)
print("Top candidates by LOF-like score:")
for source_id, score in lof_ranking[:5]:
    row = next(item for item in standardized_rows if item["spectrum_id"] == source_id)
    print(source_id, round(score, 3), row["reference_flag"], nearest_neighbors[source_id])


In [ ]:
random.seed(7)


def isolate_depth(candidate_id, subset_ids, depth=0):
    if len(subset_ids) <= 1:
        return depth

    feature_indices = []
    for index, name in enumerate(feature_names):
        values = [vectors[source_id][index] for source_id in subset_ids]
        if max(values) - min(values) > 1e-12:
            feature_indices.append(index)
    if not feature_indices:
        return depth

    feature_index = random.choice(feature_indices)
    values = [vectors[source_id][feature_index] for source_id in subset_ids]
    split_value = random.uniform(min(values), max(values))

    left_ids = [source_id for source_id in subset_ids if vectors[source_id][feature_index] < split_value]
    right_ids = [source_id for source_id in subset_ids if vectors[source_id][feature_index] >= split_value]
    next_subset = left_ids if candidate_id in left_ids else right_ids
    if len(next_subset) == len(subset_ids):
        return depth + 1
    return isolate_depth(candidate_id, next_subset, depth + 1)


all_ids = [row["spectrum_id"] for row in standardized_rows]
average_isolation_depth = {}
for source_id in all_ids:
    depths = [isolate_depth(source_id, all_ids) for _ in range(200)]
    average_isolation_depth[source_id] = sum(depths) / len(depths)

isolation_score = {source_id: 1.0 / depth for source_id, depth in average_isolation_depth.items()}
isolation_ranking = sorted(isolation_score.items(), key=lambda item: item[1], reverse=True)
print("Top candidates by simplified isolation score:")
for source_id, score in isolation_ranking[:5]:
    row = next(item for item in standardized_rows if item["spectrum_id"] == source_id)
    print(source_id, round(score, 3), row["reference_flag"], round(average_isolation_depth[source_id], 3))


In [ ]:
top_candidates = []
for ranking in [distance_ranking, lof_ranking, isolation_ranking]:
    for source_id, _ in ranking[:3]:
        if source_id not in top_candidates:
            top_candidates.append(source_id)

print("Cross-method anomaly review list:")
for source_id in top_candidates:
    row = next(item for item in standardized_rows if item["spectrum_id"] == source_id)
    print({
        "spectrum_id": source_id,
        "label": row["label"],
        "reference_flag": row["reference_flag"],
        "distance_score": round(mean_neighbor_distance[source_id], 3),
        "lof_like": round(lof_like_score[source_id], 3),
        "isolation_score": round(isolation_score[source_id], 3),
    })

print("Observation:")
print("  CR01 is the strongest candidate but it is a reduction artifact, not a new object.")
print("  EM01 and WD01 stay near the top across methods and are better follow-up targets.")


In [ ]:
try:
    from sklearn.ensemble import IsolationForest
    from sklearn.neighbors import LocalOutlierFactor
except ModuleNotFoundError:
    print("scikit-learn 未安装；已跳过标准库异常检测示例。")
else:
    feature_matrix = [feature_vector(row) for row in standardized_rows]
    lof = LocalOutlierFactor(n_neighbors=3)
    lof_predictions = lof.fit_predict(feature_matrix)
    lof_scores = [-value for value in lof.negative_outlier_factor_]
    isolation_forest = IsolationForest(random_state=7, contamination=0.16)
    isolation_forest.fit(feature_matrix)
    forest_scores = [-value for value in isolation_forest.score_samples(feature_matrix)]
    print("sklearn LOF top 5:")
    for source_id, score in sorted(zip(all_ids, lof_scores), key=lambda item: item[1], reverse=True)[:5]:
        print(source_id, round(score, 3))
    print("sklearn IsolationForest top 5:")
    for source_id, score in sorted(zip(all_ids, forest_scores), key=lambda item: item[1], reverse=True)[:5]:
        print(source_id, round(score, 3))
    print("LOF predicted outliers:", [source_id for source_id, flag in zip(all_ids, lof_predictions) if flag == -1])
